# Plotly Global Instance Explorer

This notebook demonstrates `plotly.global.instance_explorer`, a hover-only batch overview for calibrated prediction and uncertainty space. It is not a global CE explanation method. Marker size indicates how many instances share a deterministic aggregated prediction/uncertainty position.

In [1]:
import numpy as np

from calibrated_explanations import WrapCalibratedExplainer
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from crepes.extras import DifficultyEstimator

import ce_visualization_plotly.plugin  # registers Plotly styles

## Section 1: Classification

The classification section uses a 60/20/20 proper-training, calibration, and query split. The explorer aggregates nearby rounded positions with `position_precision=2`, so larger markers represent multiple instances in the same prediction/uncertainty position.

In [5]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
)
x_proper, x_tmp, y_proper, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=7, stratify=y
)
x_cal, x_query, y_cal, y_query = train_test_split(
    x_tmp, y_tmp, test_size=0.5, random_state=7, stratify=y_tmp
)

classifier = RandomForestClassifier(n_estimators=80, random_state=7)
explainer = WrapCalibratedExplainer(classifier)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True
explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

classification_explanations = explainer.explain_factual(x_query[:120])
classification_result = classification_explanations.plot(
    style="plotly.global.instance_explorer",
    task="classification",
    position_precision=2,
    show=True,
)


## Section 2: Probabilistic Regression / Thresholded Regression

Regression can be viewed probabilistically by defining a target event such as `y <= threshold`. Hover shows the event probability and calibrated probability interval.

In [6]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=7,
    n_informative=5,
    noise=8.0,
    random_state=11,
)
x_proper, x_tmp, y_proper, y_tmp = train_test_split(
    X_reg, y_reg, test_size=0.4, random_state=11
)
x_cal, x_query, y_cal, y_query = train_test_split(
    x_tmp, y_tmp, test_size=0.5, random_state=11
)

regressor = RandomForestRegressor(n_estimators=80, random_state=11)
regression_explainer = WrapCalibratedExplainer(regressor)
regression_explainer.fit(x_proper, y_proper)
assert regression_explainer.fitted is True
regression_explainer.calibrate(x_cal, y_cal)
assert regression_explainer.calibrated is True

threshold = float(np.median(y_proper))
threshold_explanations = regression_explainer.explain_factual(
    x_query[:120],
    threshold=threshold,
)
threshold_result = threshold_explanations.plot(
    style="plotly.global.instance_explorer",
    task="probabilistic_regression",
    threshold=threshold,
    position_precision=2,
    show=True,
)

## Section 3: Conformal Regression / Percentile Interval Regression

For percentile interval regression, the x-axis is the point prediction / median and the y-axis is calibrated prediction interval width. Hover reports the percentile interval metadata and calibrated prediction interval.

In [7]:
X_reg2, y_reg2 = make_regression(
    n_samples=5000,
    n_features=7,
    n_informative=5,
    noise=10.0,
    random_state=19,
)
x_proper, x_tmp, y_proper, y_tmp = train_test_split(
    X_reg2, y_reg2, test_size=0.4, random_state=19
)
x_cal, x_query, y_cal, y_query = train_test_split(
    x_tmp, y_tmp, test_size=0.5, random_state=19
)

interval_regressor = RandomForestRegressor(n_estimators=80, random_state=19)
interval_explainer = WrapCalibratedExplainer(interval_regressor)
interval_explainer.fit(x_proper, y_proper)
assert interval_explainer.fitted is True
interval_explainer.calibrate(x_cal, y_cal)
assert interval_explainer.calibrated is True

interval_explainer.set_difficulty_estimator(
        DifficultyEstimator().fit(X=x_proper, learner=interval_explainer.learner, scaler=True)
    )

percentiles = (10, 90)
interval_explanations = interval_explainer.explain_factual(
    x_query[:120],
    low_high_percentiles=percentiles,
)
interval_result = interval_explanations.plot(
    style="plotly.global.instance_explorer",
    task="conformal_regression",
    low_high_percentiles=percentiles,
    position_precision=2,
    show=True,
)
